In [ ]:
import pandas as pd
import numpy as np
import os

# ─── Chemins ────────────────────────────────────────────────────────────────
BRUTES_PATH  = r"C:\Drone_Attack_Similarity_Project\DATASET\Brutes"
ATTACKS_PATH = BRUTES_PATH + "\\Each_Attacks_CSV"
NORMAL_PATH  = BRUTES_PATH + "\\NomralAttacks"
UAVIDS_PATH  = BRUTES_PATH + "\\UAVIDS-2025"
TABLES_PATH  = r"C:\Drone_Attack_Similarity_Project\Rapport\tables"
os.makedirs(TABLES_PATH, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════
# DATASET 1 — UAVIDS-2025
# ═══════════════════════════════════════════════════════════════════════════
df_uavids = pd.read_csv(UAVIDS_PATH + "\\UAVIDS-2025.csv")

print("=" * 55)
print("DATASET 1 : UAVIDS-2025")
print("=" * 55)
print(f"Dimensions : {df_uavids.shape[0]} lignes × {df_uavids.shape[1]} colonnes")
print(f"Colonnes   : {df_uavids.columns.tolist()}")
print("\nDistribution des classes :")
print(df_uavids["label"].value_counts())
df_uavids.head()

# ═══════════════════════════════════════════════════════════════════════════
# DATASET 2 — IoT/UAV Composite  (10 fichiers → 1 seul DataFrame)
# ═══════════════════════════════════════════════════════════════════════════

# Chargement des fichiers sources avec leur label d'attaque
FICHIERS_ATTAQUES = {
    "Bruteforce"    : ATTACKS_PATH + "\\BruteforceC.csv",
    "DDoS"          : ATTACKS_PATH + "\\DDoSC.csv",
    "DoS"           : ATTACKS_PATH + "\\DoSC.csv",
    "Evil Twin"     : ATTACKS_PATH + "\\EvilC.csv",
    "Fake Landing"  : ATTACKS_PATH + "\\FakelandingC.csv",
    "MITM"          : ATTACKS_PATH + "\\MITMC.csv",
    "Reconnassiance": ATTACKS_PATH + "\\ReconnassianceC.csv",
    "Reply"         : ATTACKS_PATH + "\\ReplyC.csv",
    "Scanning"      : ATTACKS_PATH + "\\ScanningC.csv",
    "Normal"        : NORMAL_PATH  + "\\Normal.csv",
}

# Aperçu des tailles avant fusion
print("=" * 55)
print("DATASET 2 : IoT/UAV Composite — fichiers sources")
print("=" * 55)
print(f"  {'Fichier source':<18} {'Lignes':>8} {'Colonnes':>9}")
print("-" * 40)

fragments = []
resume_sources = []

for label, path in FICHIERS_ATTAQUES.items():
    df_tmp = pd.read_csv(path)
    # Si la colonne label n'existe pas encore, on l'ajoute
    if "label" not in df_tmp.columns:
        df_tmp["label"] = label
    print(f"  {label:<18} {df_tmp.shape[0]:>8} {df_tmp.shape[1]:>9}")
    resume_sources.append({"Source": label, "Lignes": df_tmp.shape[0], "Colonnes": df_tmp.shape[1]})
    fragments.append(df_tmp)

# Fusion en un seul dataset
df_composite = pd.concat(fragments, ignore_index=True)

print()
print("=" * 55)
print(f"IoT/UAV Composite FUSIONNÉ : {df_composite.shape[0]} lignes × {df_composite.shape[1]} colonnes")
print("=" * 55)
print("\nDistribution des classes :")
print(df_composite["label"].value_counts())



# ═══════════════════════════════════════════════════════════════════════════
# Résumé comparatif des 2 datasets
# ═══════════════════════════════════════════════════════════════════════════
datasets = {
    "UAVIDS-2025"       : df_uavids,
    "IoT/UAV Composite" : df_composite,
}

print("\n" + "=" * 60)
print("RÉSUMÉ — 2 datasets réels")
print("=" * 60)
print(f"  {'Dataset':<22} {'Lignes':>8} {'Colonnes':>9} {'Classes':>8}")
print("-" * 55)
for name, df in datasets.items():
    n_classes = df["label"].nunique() if "label" in df.columns else "?"
    print(f"  {name:<22} {df.shape[0]:>8} {df.shape[1]:>9} {n_classes:>8}")

# Export CSV pour rapport
shape_df = pd.DataFrame([
    {
        "Dataset"  : name,
        "Lignes"   : df.shape[0],
        "Colonnes" : df.shape[1],
        "Classes"  : df["label"].nunique() if "label" in df.columns else None,
        "Labels"   : ", ".join(df["label"].unique()) if "label" in df.columns else "",
    }
    for name, df in datasets.items()
])
shape_df.to_csv(TABLES_PATH + "\\shape_datasets.csv", index=False)
print("\nExporté : shape_datasets.csv")

# Export distribution des labels — UAVIDS-2025
df_uavids["label"].value_counts().rename_axis("Attaque").reset_index(
    name="Nb_Echantillons"
).to_csv(TABLES_PATH + "\\label_distribution_UAVIDS.csv", index=False)

# Export distribution des labels — Composite
df_composite["label"].value_counts().rename_axis("Attaque").reset_index(
    name="Nb_Echantillons"
).to_csv(TABLES_PATH + "\\label_distribution_Composite.csv", index=False)

# Export résumé des fichiers sources
pd.DataFrame(resume_sources).to_csv(TABLES_PATH + "\\sources_composite.csv", index=False)

print("Exporté : label_distribution_UAVIDS.csv")
print("Exporté : label_distribution_Composite.csv")
print("Exporté : sources_composite.csv")

shape_df





DATASET 1 : UAVIDS-2025
Dimensions : 122171 lignes × 23 colonnes
Colonnes   : ['FlowID', 'FlowDuration/s', 'SrcAddr', 'SrcPort', 'DstAddr', 'DstPort', 'Protocol', 'TxPackets', 'RxPackets', 'LostPackets', 'TxBytes', 'RxBytes', 'TxPacketRate/s', 'RxPacketRate/s', 'TxByteRate/s', 'RxByteRate/s', 'MeanDelay/s', 'MeanJitter/s', 'Throughput/Kbps', 'MeanPacketSize', 'PacketDropRate', 'AverageHopCount', 'label']

Distribution des classes :
label
Normal Traffic      26172
Blackhole Attack    26110
Wormhole Attack     26086
Sybil Attack        24077
Flooding Attack     19726
Name: count, dtype: int64
DATASET 2 : IoT/UAV Composite — fichiers sources
  Fichier source       Lignes  Colonnes
----------------------------------------


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Drone_Attack_Similarity_Project\\DATASET\\Brutes\\Each_Attacks_CSV\\BruteforceC.csv'

: 